## Day 14 — Critical Finding: Look-Ahead Bias

Yesterday's backtest (Day 13) contained a methodological flaw: the Vol Filter 
strategy used the SAME day's volatility reading to decide that SAME day's 
position, then captured that SAME day's full return. This is unrealistic, 
since volatility can only be measured AFTER a trading day closes — you 
cannot trade on information you don't have yet.

### What we tested

We compared three different timing/execution assumptions:

1. **Same-day (biased):** original Day 13 version — uses today's signal 
   to capture today's return
2. **Next-day, close-to-close:** signal shifted by 1 day, return still 
   measured close-to-close
3. **Next-day, open-to-close:** signal shifted by 1 day, return measured 
   from market open to close (excludes overnight gap)

### Results

| Strategy | Same-day (biased) | Next-day, close-to-close | Next-day, open-to-close |
|---|---|---|---|
| Vol Filter | Sharpe 1.6793, final 11252.17 | Sharpe 0.6648, final 10482.04 | Sharpe -0.6018, final 9576.75 |
| Buy-the-Dip | Sharpe 1.2098, final 12209.74 | Sharpe 1.2081, final 12201.48 | Sharpe ~1.21, final 12218.08 |

*(All final values in USD, starting capital $10,000)*

### Key insight

**Vol Filter's apparent edge was almost entirely an artifact of look-ahead bias.** 
Its Sharpe ratio swings wildly — from 1.68 down to -0.60 — depending purely on 
timing/execution assumptions, with no change to the underlying strategy logic. 
This reveals the original result was fragile and not trustworthy.

**Buy-the-Dip remained stable** (~$12,200 final value, Sharpe ~1.21) across all 
three methodologies. This robustness comes from a structural difference: 
Buy-the-Dip accumulates shares over time (a one-time purchase decision per 
signal), while Vol Filter directly multiplies a signal against a same-day 
return — making it far more sensitive to exactly which return window is measured.

### Lesson for future backtesting

Always check: 
1. Does my signal use only information available BEFORE the trade executes?
2. Is my result sensitive to small changes in execution timing assumptions?

If a strategy's performance collapses under minor timing adjustments, the 
original "edge" likely wasn't real.

In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

raw = yf.download("SPY", start="2025-01-01", end="2026-01-01")
data = raw.droplevel("Ticker", axis=1)
data['daily_return'] = data['Close'].pct_change()

data['rolling_vol'] = (data['daily_return'].rolling(21)).std() * np.sqrt(252)
vol_threshold = data['rolling_vol'].median()
data['signal'] = np.where(data['rolling_vol'] < vol_threshold, 1, 0)
data['strategy_return'] = data['signal'] * data['daily_return']
strategy_sharpe = (data['strategy_return'].mean() / data['strategy_return'].std()) * np.sqrt(252)

data['signal_realistic'] = data['signal'].shift(1)
data['strategy_return_realistic'] = data['signal_realistic'] * data['daily_return']

strategy_realistic_sharpe = (data['strategy_return_realistic'].mean() / 
                             data['strategy_return_realistic'].std()) * np.sqrt(252)

print(f"Original (biased) Sharpe: {strategy_sharpe:.4f}")
print(f"Realistic (shifted) Shrape: {strategy_realistic_sharpe:.4f}")

final_original = (1 + data['strategy_return'].fillna(0)).cumprod().iloc[-1]
final_realistic = (1 + data['strategy_return_realistic'].fillna(0)).cumprod().iloc[-1]
print(f"Original (biased) final value: ${final_original*10000:.2f}")
print(f"Realistic final value ${final_realistic*10000:.2f}")



In [21]:
data['open_return'] = (data['Close'] - data['Open']) / data['Open']
data['strategy_return_open_exec'] = data['signal_realistic'] * data['open_return']

strategy_open_sharpe = (data['strategy_return_open_exec'].mean() / 
                        data['strategy_return_open_exec'].std()) * np.sqrt(252)

final_open = (1 + data['strategy_return_open_exec'].fillna(0)).cumprod().iloc[-1]

print(f"Close-to-close Sharpe: {strategy_realistic_sharpe:.4f}")
print(f"Open-to-close Sharpe: {strategy_open_sharpe:.4f}")
print(f"Close-to-close final: ${final_realistic*10000:.2f}")
print(f"Open-to-close final: ${final_open*10000:.2f}")

Close-to-close Sharpe: 0.6648
Open-to-close Sharpe: -0.6018
Close-to-close final: $10482.04
Open-to-close final: $9576.75
